# hdrs

> App integration: the headers that carry the registry's assets

In [1]:
#| default_exp hdrs

An app integrates answerui by passing `reg.hdrs()` and `reg.ftrs()` to `FastHTML` (or `fast_app`). The headers are the stylesheet and the behavior script. The footer is the sprite sheet for icons. The registry can change after that call, so the headers are lazy: each response renders them from the current state of the registry.

In [2]:
from fastcore.utils import *
from fasthtml.common import Style, Script, Link
from answerui.registry import Registry
import answerui.behaviors  # installs Registry.build_js
from fastlucide.icons import spritesheet
import hashlib
from starlette.responses import Response

In [3]:
from fasthtml.common import *
from answerui.registry import reg, hypelt
from fastcore.test import *
from fastlucide.icons import Check

Building the stylesheet walks every rule, and the result can only change when the registry does. The registry counts its changes in `version`, so `assets` builds once per version and keeps the stylesheet and script with a hash of each, the name a served asset carries:

In [4]:
def _hash(s): return hashlib.sha256(s.encode()).hexdigest()[:12]

@patch
def assets(self:Registry):
    "The stylesheet and script with their content hashes, keyed by extension, built once per registry version"
    if getattr(self, '_assets_version', None) != self.version:
        self._assets = {ext: AttrDict(body=body, hash=_hash(body)) for ext,body in (('css', self.build_css()), ('js', self.build_js()))}
        self._assets_version = self.version
    return self._assets


The same build comes back until a rule changes:

In [5]:
r = Registry()
built = r.assets()
test_is(r.assets(), built)
r.rule('page', 'main { max-width: 65ch; }')
assert r.assets() is not built
r.assets()['css'].hash

'ee68738f88fc'

When a FastHTML page is rendered, any object with an `__ft__` method is asked for its markup at that moment. We make use of this: `_Lazy` wraps a function to make it get called per response, and a registry's `hdrs` returns its stylesheet and behavior script that way. The headers belong to the registry, so a registry of one's own renders its own page with nothing else changed:

In [6]:
class _Lazy:
    def __init__(self, f): self.f = f
    def __ft__(self): return self.f()

@patch
def hdrs(self:Registry):
    "Headers for `fast_app(hdrs=...)`: the stylesheet and behavior script, rendered from this registry per response"
    return _Lazy(lambda: Style(self.assets()['css'].body)), _Lazy(lambda: Script(self.assets()['js'].body))


To see the deferral, build the headers first and define a component afterwards. Rendering the stylesheet header then includes it.

In [7]:
hdrs = reg.hdrs()
LateCard = hypelt('LateCard', css='late-card { display: block; color: teal; }', layer='components',
    on={('&', 'click'): r"el.remove()"}, doc='Defined after the app; click to remove.')
css = to_xml(hdrs[0])
print(css[css.index('/* LateCard.base */'):][:60])

/* LateCard.base */
late-card { display: block; color: teal;


Redefining it changes the next render. There is no rebuild step and nothing to invalidate:

In [8]:
LateCard = hypelt('LateCard', css='late-card { display: block; color: navy; }', layer='components', doc='Defined after the app.')
css = to_xml(hdrs[0])
assert 'navy' in css and 'teal' not in css

An icon is a reference to a symbol in fastlucide's `spritesheet`. When a module imports an icon from `fastlucide.icons`, the name is added to the sheet. The sheet is an `svg`, so it must be in the body, after the content that uses it. FastHTML renders `ftrs` at the end of the body, and `ftrs` returns the sheet for that purpose. `spritesheet` renders itself when the page does, so it needs no `_Lazy` wrapper:

In [9]:
@patch
def ftrs(self:Registry):
    "Footers for `fast_app(ftrs=...)`: fastlucide's sprite sheet, with every icon that any module has imported"
    return (spritesheet,)

## A whole page

That is all an app needs. FastHTML's test client fetches a page without starting a server. The head of the response has htmx, then our stylesheet and script. The sprite sheet is at the end of the body, after the icon that uses it.

In [10]:
app = FastHTML(hdrs=reg.hdrs(), ftrs=reg.ftrs())
rt = app.route

@rt
def index(): return Titled('Hello', LateCard(Check(), 'Defined after the app.'))

page = Client(app).get('/').text
assert page.index('<symbol id="lc-check"') > page.index('late-card')
print(re.sub(r'(?s)(<(style|script)>).*?(?=</\2>)', r'\1…', page))

<!doctype html>
<html>
  <head>
    <title>Hello</title>
    <link rel="canonical" href="https://testserver/">
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<script src="https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js"></script><script src="https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.13/fasthtml.js"></script><script src="https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js"></script><script src="https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js"></script>    <style>…</style>
<script>…</script>  </head>
  <body>
<main class="container">      <h1>Hello</h1>
<late-card><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" class="lucide-icon "><use href="#lc-check"></use></svg>Defined after the app.</late-card></main>    <style>…</style>
<svg xmlns="http://www.w3.org/2000/svg" style="display: none"><defs><symbol id="lc-check"><path d="M20

`fast_app` works the same way, but it adds Pico CSS by default, so we turn that off.

In [11]:
app, rt = fast_app(hdrs=reg.hdrs(), ftrs=reg.ftrs(), pico=False)
@rt
def index(): return Titled('Hello', LateCard('Defined after the app.'))

assert 'pico' not in (r:= Client(app).get('/').text)
assert 'late-card' in r

## Serving the assets

Inline headers send the whole stylesheet and script with every response. `serve` instead mounts them at URLs named by a hash of their content. The browser fetches each once and keeps it. The headers `serve` adds are lazy, like those of `hdrs`: each response links the current build under its hash. While the registry is unchanged, every page links the same name. When a rule changes, the next page links a new name and the browser fetches it. `serve` remembers every build it has linked. Its route answers a linked name with that build, marked immutable, and any other name with 404. Because browsers and proxies keep an immutable response for a year, a name never changes what it returns. `serve` also adds `ftrs()` to the app. It returns the two headers, for an app that builds its own `head`.

In [12]:
@patch
def serve(
    self:Registry,
    app,              # The FastHTML app to mount the route on and add the headers to
    path='/answerui', # URL prefix the assets are served under
):
    "Serve the stylesheet and script under content-hashed names and add the sprite sheet to `app`; returns the two headers, for an app that builds its own head"
    served, mimes = {}, {'css': 'text/css', 'js': 'text/javascript'}
    def link(ext):
        a = self.assets()[ext]
        served[f'{a.hash}.{ext}'] = (a.body, mimes[ext])
        return f'{path}/{a.hash}.{ext}'
    @app.get(f'{path}/{{name}}')
    def asset(name: str):
        if name not in served: return Response(status_code=404)
        body, mime = served[name]
        return Response(body, media_type=mime, headers={'Cache-Control': 'public, max-age=31536000, immutable'})
    hdrs = _Lazy(lambda: Link(rel='stylesheet', href=link('css'))), _Lazy(lambda: Script(src=link('js')))
    app.hdrs, app.ftrs = [*app.hdrs, *hdrs], [*app.ftrs, *self.ftrs()]
    return hdrs

In [13]:
app = FastHTML()
rt = app.route
reg.serve(app)

@rt
def index(): return Titled('Hello', LateCard('Defined after the app.'))

page = Client(app).get('/').text
href = re.search(r'href="(/answerui/[^"]+\.css)"', page)[1]
asset = Client(app).get(href)
test_eq(asset.headers['cache-control'], 'public, max-age=31536000, immutable')
assert 'navy' in asset.text
href

'/answerui/523580fc6bd8.css'

A change to the registry gives the next page a new name. The old name still serves the build it was linked with, and a name that was never linked is not found:

In [14]:
LateCard = hypelt('LateCard', css='late-card { display: block; color: teal; }', layer='components', doc='Defined after the app.')
href2 = re.search(r'href="(/answerui/[^"]+\.css)"', Client(app).get('/').text)[1]
assert href2 != href
assert 'navy' in Client(app).get(href).text
assert 'teal' in Client(app).get(href2).text
test_eq(Client(app).get('/answerui/000000000000.css').status_code, 404)

## Export

In [15]:
#| hide
import nbdev; nbdev.nbdev_export()